In [ ]:
from pathlib import Path
import pickle

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from topostats.damage.io import construct_grains_collection_from_topostats_files
from topostats.damage.classes import UnanalysedGrainCollection, GrainCollection
from topostats.damage.damage_processing import find_defects_in_height_and_curvature

# Config

In [ ]:
dir_base = Path("/Volumes/shared/pyne_group/Shared/AFM_Data/dna_damage/Cs137_irradiations")
dir_remote_analysis = dir_base / "20260204-analysis-getting-back-into-the-project"
threshold_contour_length = 300  # Cutoff for the minimum contour length for grains to consider
dir_results = dir_remote_analysis / "analysis_results"

local_analysis: bool = True  # Whether to do the analysis using a locally saved cache file.
force_reload_from_file: bool = False  # Whether to force-reload the data from the original topostats files
dir_local_pkl_cache = Path("/Users/sylvi/topo_data/dna_damage_cache")  # Where to load/save the local cache file

if local_analysis:
    dir_results = dir_local_pkl_cache / "analysis_results"  # Where to save the results of the analysis
else:
    dir_results = dir_remote_analysis / "output" / "analysis_results"  # Where to save the results of the analysis

# defects
height_defect_method = "iqr"
height_threshold_iqr_multiplier = 3.0
height_threshold_absolute_nm = 0.8

curvature_defect_method = "iqr"
curvature_threshold_iqr_multiplier = 3.0
curvature_threshold_absolute_pernm = 0.1

connect_close_defect_threshold_nm = 10

# plotting
folder_to_labels = {
    "Controls": "Control",
    "MilliQ/5_percent_damage": "MilliQ 5%",
    "MilliQ/20_percent_damage": "MilliQ 20%",
    "MilliQ/50_percent_damage": "MilliQ 50%",
    "TE/5_percent_damage": "TE 5%",
    "TE/20_percent_damage": "TE 20%",
    "TE/50_percent_damage": "TE 50%",
}
plotting_folder_order = [
    "Controls",
    "MilliQ/5_percent_damage",
    "MilliQ/20_percent_damage",
    "MilliQ/50_percent_damage",
    "TE/5_percent_damage",
    "TE/20_percent_damage",
    "TE/50_percent_damage",
]

In [ ]:
def update_x_labels(ax: plt.Axes):
    # Update x-axis folder labels to be more readable
    x_labels = [item.get_text() for item in ax.get_xticklabels()]
    new_labels = [folder_to_labels.get(label, label) for label in x_labels]
    ax.set_xticklabels(new_labels, rotation=45, ha="right")

# Loading data

In [ ]:
assert dir_local_pkl_cache.exists()
dir_results.mkdir(exist_ok=True)
assert dir_results.exists()
if not local_analysis:
    assert dir_base.exists()
    assert dir_remote_analysis.exists()
    dir_processed_data = dir_remote_analysis / "output"
    assert dir_processed_data.exists()

    topo_files = list(dir_processed_data.glob("*/**/processed/*.topostats"))
    print(f"found {len(topo_files)} topo files")

    # Load the corresponding csv file
    csv_grain_stats: Path = dir_processed_data / "grain_statistics.csv"
    assert csv_grain_stats.exists()
    df_grain_stats = pd.read_csv(csv_grain_stats)
    print(f"grain stats columns: {df_grain_stats.columns}")

    # convert some columns to nanometres
    df_grain_stats["total_contour_length"] /= 1e-9

    # convert basename to only use the parent directory
    df_grain_stats["basename"] = df_grain_stats["basename"].apply(lambda x: Path(x).parent)

    # Strip the file extension from the "image" column
    df_grain_stats["image"] = df_grain_stats["image"].apply(lambda x: Path(x).stem)

    # replace NaN in "num_crossings" with 0
    df_grain_stats["num_crossings"] = df_grain_stats["num_crossings"].fillna(0)

    # drop any row where the basename contains "diabolical tips"
    df_grain_stats = df_grain_stats[~df_grain_stats["basename"].astype(str).str.contains("diabolical tips")]

    print(df_grain_stats["basename"].value_counts())

# Sample vetting

In [ ]:
if not local_analysis:
    # plot contour length distributions
    sns.stripplot(data=df_grain_stats, x="basename", y="total_contour_length", s=2)
    sns.violinplot(data=df_grain_stats, x="basename", y="total_contour_length", inner=None)
    plt.xticks(rotation=90)
    plt.title("Contour length distributions")
    plt.show()

    # drop any rows with contour length less than a threshold
    n_rows_before = len(df_grain_stats)
    df_grain_stats = df_grain_stats[df_grain_stats["total_contour_length"] >= threshold_contour_length]
    n_rows_after = len(df_grain_stats)
    print(
        f"dropped {n_rows_before - n_rows_after} rows with contour length < {threshold_contour_length} nm."
        f"remaining rows: {n_rows_after}"
    )

    sns.stripplot(data=df_grain_stats, x="basename", y="total_contour_length", s=2)
    sns.violinplot(data=df_grain_stats, x="basename", y="total_contour_length", inner=None)
    plt.xticks(rotation=90)
    plt.title("Contour length distributions")
    plt.show()

# Construct initial graincollection

In [ ]:
if not local_analysis:
    unanalysed_grain_collection: UnanalysedGrainCollection = construct_grains_collection_from_topostats_files(
        dir_to_save_cache_files=dir_results,
        dir_processed_data=dir_processed_data,
        df_grain_stats=df_grain_stats,
        bbox_padding=1,
        force_reload=force_reload_from_file,
    )
    print(unanalysed_grain_collection)

    # Save the object to a pickle file
    pkl_file = dir_local_pkl_cache / "unanalysed_grain_collection.pkl"
    with open(pkl_file, "wb") as f:
        pickle.dump(unanalysed_grain_collection, f)
    # save a cache of the csv file too
    df_grain_stats.to_csv(dir_local_pkl_cache / "grain_statistics.csv", index=False)
else:
    # Load the object from the pickle file
    pkl_file = dir_local_pkl_cache / "unanalysed_grain_collection.pkl"
    with open(pkl_file, "rb") as f:
        unanalysed_grain_collection: UnanalysedGrainCollection = pickle.load(f)
    print(unanalysed_grain_collection)
    # load the cached csv file too
    df_grain_stats = pd.read_csv(dir_local_pkl_cache / "grain_statistics.csv")

# Create new analysed data model

In [ ]:
grain_collection: GrainCollection = GrainCollection.from_unanalysed_grain_collection(
    unanalysed_collection=unanalysed_grain_collection,
)

print(grain_collection)

# Defect detection

In [ ]:
bad_grains = find_defects_in_height_and_curvature(
    grain_collection=grain_collection,
    height_defect_method=height_defect_method,
    height_threshold_iqr_multiplier=height_threshold_iqr_multiplier,
    height_threshold_absolute_nm=height_threshold_absolute_nm,
    curvature_defect_method=curvature_defect_method,
    curvature_threshold_iqr_multiplier=curvature_threshold_iqr_multiplier,
    curvature_threshold_absolute_pernm=curvature_threshold_absolute_pernm,
    connect_close_defect_threshold_nm=connect_close_defect_threshold_nm,
)

# remove bad grains
grain_collection.remove_grains(bad_grains)

print(grain_collection)

# Plot defect quantification

In [ ]:
# dataframe with columns of grain_id, folder, num_height_defects, num_curvature_defects
height_curvature_defect_counts = []
for grain_id, grain in grain_collection.grains.items():
    folder = grain.folder
    num_height_defects = grain.height_defect_data.num_defects
    num_curvature_defects = grain.curvature_defect_data.num_defects
    height_curvature_defect_counts.append(
        {
            "grain_id": grain_id,
            "folder": folder,
            "num_height_defects": num_height_defects,
            "num_curvature_defects": num_curvature_defects,
            "image": grain.filename,
            "grain_number": grain.file_grain_id,
        }
    )
df_defect_counts = pd.DataFrame(height_curvature_defect_counts)

# Violin plot
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
sns.violinplot(
    data=df_defect_counts, x="folder", y="num_height_defects", inner=None, ax=ax[0], order=plotting_folder_order
)
sns.stripplot(
    data=df_defect_counts, x="folder", y="num_height_defects", color="k", size=2, ax=ax[0], order=plotting_folder_order
)
sns.violinplot(
    data=df_defect_counts, x="folder", y="num_curvature_defects", inner=None, ax=ax[1], order=plotting_folder_order
)
sns.stripplot(
    data=df_defect_counts,
    x="folder",
    y="num_curvature_defects",
    color="k",
    size=2,
    ax=ax[1],
    order=plotting_folder_order,
)
update_x_labels(ax[0])
update_x_labels(ax[1])
ax[0].set_title("Height defects per grain")
ax[1].set_title("Curvature defects per grain")
plt.tight_layout()
plt.show()


sampled_grains = grain_collection.sample(n=1, seed=0)
for grain in sampled_grains.grains.values():
    grain.plot(mask_alpha=0.1, curvature_defects=True, linemode="curvature")

# Merge stats into grainstats

In [ ]:
# Create a copy of the grain_statistics dataframe
df_grain_stats_copy = df_grain_stats.copy()

df_grain_stats_merged = df_grain_stats_copy.merge(df_defect_counts, on=["image", "grain_number"])

# save to csv
df_grain_stats_merged.to_csv(dir_results / "defect_grain_statistics.csv", index=False)